# 03 Modelo Clasificacion Ticket Alto

Objetivo: predecir si un ticket es `ticket_alto` a partir de las variables reconstruidas desde el OLAP.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]
PARQUET_PATH = PROJECT_ROOT / "parquets" / "01_Carga_y_Validacion_Parquet" / "01_base_tickets_modelado.parquet"
df = pd.read_parquet(PARQUET_PATH)
df.head()

In [ ]:
target = "ticket_alto"
columnas_excluir = ["id_ticket_modelado", "fecha", "total_pedido", "monto_pago", "ticket_alto"]
X = df.drop(columns=columnas_excluir)
y = df[target]
X.shape, y.shape

In [ ]:
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numericas
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categoricas
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Modelo base sugerido
modelo = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Alternativa sugerida: RandomForestClassifier()
modelo